# Clase 04 — Representaciones dispersas

**Curso:** NLP Clásico con Python · **Bloque 2:** Representar y clasificar
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/04_representaciones_dispersas.md`

## Qué vas a ver en esta clase

- La **bolsa de palabras** (*bag of words*, BoW): del texto a un vector de recuentos, primero **a mano** y después con `CountVectorizer`.
- Los **n-gramas** de palabras y de caracteres: qué recuperan y cuánto hacen crecer el vocabulario.
- **TF-IDF**: cálculo a mano sobre un corpus mínimo, la fórmula exacta de scikit-learn y en qué se distingue de la del libro de Manning y colaboradores.
- La **dispersión**: matrices `scipy.sparse`, memoria y efecto de `min_df`, `max_df` y `max_features`.
- La **similitud del coseno** y un experimento guiado: recuperar el «gemelo limpio» de cada reseña con ruido.
- Los **límites** de estas representaciones y por qué el vectorizador se ajusta (`fit`) solo con los datos de entrenamiento.

## Requisitos

- Tokenizar con `re.findall(r"\w+", texto)` y nociones básicas de stopwords, stemming y lematización (clases 2 y 3).
- Python con `numpy` y `pandas`. Librerías: `scikit-learn`, `scipy`, `matplotlib`. La celda siguiente instala solo lo que falte (funciona igual en Google Colab).
- La carpeta `datos/` del curso (reseñas en español y en italiano). En Colab, súbela desde el panel de archivos.

> **Cómo usar este notebook:** ejecuta las celdas en orden. Los cálculos a mano se comprueban siempre con scikit-learn mediante un `assert`. Las cifras son de corpus pequeños y sintéticos escritos para el curso (*ejemplo ilustrativo*): no se generalizan.

---
## Instalación

Instala solo lo que falte. En un entorno con todo instalado no hace nada.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("scipy", "scipy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla**, definimos `cargar_datos` (busca `datos/`, `../datos/` y `/content/datos/`) y leemos las 300 reseñas de cada idioma. La columna `texto` es el texto **bruto** con ruido (mayúsculas, acentos perdidos, erratas, emojis…) y `texto_limpio` es su versión limpia de referencia.

In [ ]:
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

random.seed(42)                          # semilla fija: todos obtenemos los mismos resultados
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
resenas = {idioma: pd.read_csv(cargar_datos(archivo)) for idioma, archivo in ARCHIVO_RESENAS.items()}
print({idioma: df.shape for idioma, df in resenas.items()})
print("Datos y librerías cargados correctamente.")

---
## 1. Bolsa de palabras a mano

Una bolsa de palabras describe cada documento con **cuántas veces aparece cada palabra del vocabulario**, sin orden. Para verlo entero construimos la matriz término-documento de un corpus **mínimo de cinco frases** en cada idioma (*ejemplo ilustrativo*). Los pasos: tokenizar, reunir el vocabulario ordenado y contar.

In [ ]:
# datos-es
CORPUS = {
    "spa": ["el envío llegó rápido",
           "el envío llegó tarde y muy tarde",
           "el producto no está bien",
           "el producto está muy bien",
           "el producto llegó roto"],
    "ita": ["la spedizione è arrivata veloce",
           "la spedizione è arrivata tardi e molto tardi",
           "il prodotto non è buono",
           "il prodotto è molto buono",
           "il prodotto è arrivato rotto"],
}
STOP = {"spa": ["el", "y", "muy"], "ita": ["il", "la", "e", "molto"]}

In [ ]:
def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())

def bolsa_a_mano(corpus):
    """Devuelve (vocabulario ordenado, matriz de recuentos documentos × términos)."""
    tokens = [tokenizar(doc) for doc in corpus]
    vocabulario = sorted({t for doc in tokens for t in doc})
    indice = {t: i for i, t in enumerate(vocabulario)}
    matriz = np.zeros((len(corpus), len(vocabulario)), dtype=int)
    for i, doc in enumerate(tokens):
        for t, n in Counter(doc).items():
            matriz[i, indice[t]] = n
    return vocabulario, matriz

for idioma in ("spa", "ita"):
    vocabulario, matriz = bolsa_a_mano(CORPUS[idioma])
    print(f"Idioma: {idioma} · documentos: {matriz.shape[0]} · vocabulario: {matriz.shape[1]}")
    print(pd.DataFrame(matriz, columns=vocabulario, index=[f"d{i+1}" for i in range(len(matriz))]))
    print()

**¿Qué observas?** Cada fila es un documento y cada columna una palabra del vocabulario. `tarde` vale 2 en el documento 2 (`tarde y muy tarde`): es un **recuento**, no un 0/1. Fíjate también en que `el` aparece en las cinco frases y en que casi todas las celdas son ceros: aquí ya se intuye la dispersión (sección 5).

Ahora comprobamos la construcción a mano con `CountVectorizer`. Ojo con el patrón por defecto: descarta los tokens de **un solo carácter** (`y`, `e`, `è`).

In [ ]:
for idioma in ("spa", "ita"):
    corpus = CORPUS[idioma]
    vocabulario, matriz = bolsa_a_mano(corpus)

    cv = CountVectorizer(token_pattern=r"\w+")          # el mismo criterio que nuestra tokenización
    X_cv = cv.fit_transform(corpus)
    assert list(cv.get_feature_names_out()) == vocabulario
    assert (X_cv.toarray() == matriz).all()

    cv_defecto = CountVectorizer()                       # patrón por defecto
    cv_defecto.fit(corpus)
    perdidas = sorted(set(vocabulario) - set(cv_defecto.get_feature_names_out()))
    print(f"{idioma}: coincide con scikit-learn ✔. Con el patrón por defecto el vocabulario pasa de {len(vocabulario)} a {len(cv_defecto.vocabulary_)} (se pierden {perdidas})")

### Los parámetros de `CountVectorizer`

Casi todo lo que hiciste a mano (minúsculas, patrón de token, stopwords) es un parámetro. Comparamos el **tamaño del vocabulario** que resulta de cada opción sobre el corpus mínimo. Las stopwords de este ejemplo son una lista propia de tres o cuatro palabras (los listados completos, en la clase 3).

In [ ]:
def tamano_vocabulario(corpus, **parametros):
    parametros.setdefault("token_pattern", r"\w+")
    return len(CountVectorizer(**parametros).fit(corpus).vocabulary_)

filas = []
for etiqueta, parametros in [
    ("base (token_pattern=\\w+)", {}),
    ("stop_words (lista propia)", "stop"),
    ("min_df=2", {"min_df": 2}),
    ("max_df=0.8", {"max_df": 0.8}),
    ("max_features=5", {"max_features": 5}),
    ("ngram_range=(1, 2)", {"ngram_range": (1, 2)}),
]:
    fila = {"opción": etiqueta}
    for idioma in ("spa", "ita"):
        p = {"stop_words": STOP[idioma]} if parametros == "stop" else parametros
        fila[idioma] = tamano_vocabulario(CORPUS[idioma], **p)
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** `min_df=2` deja solo las palabras que aparecen en al menos dos documentos (en este corpus, 7 en español y 8 en italiano). `max_df=0.8` elimina las que aparecen en más del 80 % de los documentos: aquí solo `el` en español y `è` en italiano (presentes en las cinco frases). `ngram_range=(1, 2)` multiplica el vocabulario más de dos veces. Ninguna de estas opciones es «la correcta»: dependen de la tarea.

---
## 2. N-gramas de palabras y de caracteres

### 2.1 Bigramas: lo que una bolsa pura pierde

Las dos frases siguientes tienen **exactamente las mismas palabras** y significados opuestos. Para una bolsa de palabras son el mismo vector; los bigramas ven la diferencia.

In [ ]:
# datos-es
FRASES_OPUESTAS = {
    "spa": ("no es bueno, es malo", "es bueno, no es malo"),
    "ita": ("non è buono, è cattivo", "è buono, non è cattivo"),
}
PARES_ERRATA = {"spa": ("recomiendo", "recomeindo"), "ita": ("quelli", "qeulli")}

In [ ]:
for idioma, (a, b) in FRASES_OPUESTAS.items():
    for rango in [(1, 1), (2, 2), (1, 2)]:
        X_par = CountVectorizer(token_pattern=r"\w+", ngram_range=rango).fit_transform([a, b])
        print(f"{idioma} ngram_range={rango}: coseno = {cosine_similarity(X_par)[0, 1]:.4f}  ({X_par.shape[1]} rasgos)")

cv_bi = CountVectorizer(token_pattern=r"\w+", ngram_range=(2, 2)).fit(FRASES_OPUESTAS["spa"])
print("Bigramas (español):", list(cv_bi.get_feature_names_out()))

**¿Qué observas?** Con unigramas el coseno es exactamente 1 (vectores idénticos); con bigramas baja a 0,75 y con unigramas más bigramas a 0,909. Los bigramas capturan `no es`, `es bueno`, `bueno no`…: el orden local. En el corpus mínimo, `no está bien` produce los bigramas `no está` y `está bien`. **No** resuelven la negación en general (`no me ha parecido nada bueno` tiene el `no` lejos de `bueno`), pero recuperan parte de la información de orden.

### 2.2 N-gramas de caracteres: robustez ante erratas

Con `analyzer="char_wb"` el vectorizador trocea **dentro de cada palabra** (con un espacio de relleno a cada lado). Una errata cambia una palabra entera (para la bolsa de palabras es otro token), pero solo cambia **algunos** de sus n-gramas de caracteres.

In [ ]:
for idioma, (correcta, errata) in PARES_ERRATA.items():
    analizador = CountVectorizer(analyzer="char_wb", ngram_range=(3, 3)).build_analyzer()
    a, b = set(analizador(correcta)), set(analizador(errata))
    X_pal = CountVectorizer(token_pattern=r"\w+").fit_transform([correcta, errata])
    X_car = CountVectorizer(analyzer="char_wb", ngram_range=(3, 3)).fit_transform([correcta, errata])
    print(f"{idioma}: {correcta!r} frente a {errata!r}")
    print(f"   trigramas de caracteres compartidos: {len(a & b)} de {len(a)} → {sorted(a & b)}")
    print(f"   coseno con palabras = {cosine_similarity(X_pal)[0, 1]:.3f} · coseno con trigramas de caracteres = {cosine_similarity(X_car)[0, 1]:.3f}")

**¿Qué observas?** Para la bolsa de palabras `recomiendo` y `recomeindo` no tienen nada en común (coseno 0). Con trigramas de caracteres comparten seis de diez (`rec`, `eco`, `com`, `ndo`…). En italiano la errata `qeulli` mueve casi todo y solo comparten `lli` y `li ` (coseno 0,333): los caracteres **ayudan, no hacen milagros**.

### 2.3 El precio: el vocabulario crece

Contamos las características que genera cada configuración sobre las 300 reseñas **brutas** de cada idioma.

In [ ]:
configuraciones = {
    "palabras (1, 1)": dict(token_pattern=r"\w+"),
    "palabras (1, 2)": dict(token_pattern=r"\w+", ngram_range=(1, 2)),
    "palabras (1, 2), min_df=2": dict(token_pattern=r"\w+", ngram_range=(1, 2), min_df=2),
    "caracteres char_wb (3, 5)": dict(analyzer="char_wb", ngram_range=(3, 5)),
}
filas = []
for nombre, parametros in configuraciones.items():
    fila = {"configuración": nombre}
    for idioma, df in resenas.items():
        fila[idioma] = len(CountVectorizer(**parametros).fit(df["texto"]).vocabulary_)
    filas.append(fila)
crecimiento = pd.DataFrame(filas)
print(crecimiento.to_string(index=False))

**¿Qué observas?** Con solo 300 textos, pasar de palabras sueltas a palabras más bigramas multiplica el vocabulario por 3,4 (1 613 → 5 525 en español). Los caracteres (3 a 5) llegan a más de 13 000. La mayoría de esos n-gramas aparecen una sola vez: `min_df=2` recorta el vocabulario (1, 2) de 5 525 a 1 009. Cada n-grama nuevo es una columna más de la matriz.

---
## 3. TF-IDF a mano

Un recuento da solo sobrevalora palabras como `el`, presentes en todas partes. TF-IDF pondera cada palabra por **cuánto aparece en el documento (tf)** y por **lo rara que es en el corpus (idf)**.

La fórmula de **scikit-learn** (con `smooth_idf=True`, el valor por defecto) es:

- `idf(t) = ln((1 + n) / (1 + df(t))) + 1`, con `n` = número de documentos y `df(t)` = número de documentos que contienen `t`.
- `tf-idf(t, d) = tf(t, d) · idf(t)`, con `tf` = recuento bruto.
- Después, **cada fila se normaliza con la norma L2** (`norm="l2"`, valor por defecto): la suma de cuadrados de la fila vale 1.

Lo implementamos a mano y lo comprobamos con `TfidfVectorizer`.

In [ ]:
def tfidf_a_mano(corpus):
    """tf-idf de scikit-learn: idf suavizado y filas normalizadas con L2."""
    vocabulario, conteos = bolsa_a_mano(corpus)
    n = len(corpus)
    df = (conteos > 0).sum(axis=0)                      # documentos que contienen cada palabra
    idf = np.log((1 + n) / (1 + df)) + 1
    bruto = conteos * idf                               # tf · idf
    normas = np.sqrt((bruto ** 2).sum(axis=1, keepdims=True))
    return vocabulario, conteos, df, idf, bruto, bruto / normas

for idioma in ("spa", "ita"):
    corpus = CORPUS[idioma]
    vocabulario, conteos, df, idf, bruto, tfidf = tfidf_a_mano(corpus)
    tv = TfidfVectorizer(token_pattern=r"\w+")
    X_tv = tv.fit_transform(corpus)
    assert np.allclose(X_tv.toarray(), tfidf)
    assert np.allclose(tv.idf_, idf)
    assert np.allclose(np.sqrt((tfidf ** 2).sum(axis=1)), 1.0)
    print(idioma, "coincide con TfidfVectorizer ✔")

# Tabla del documento 1 (español)
vocabulario, conteos, df, idf, bruto, tfidf = tfidf_a_mano(CORPUS["spa"])
n = len(CORPUS["spa"])
presentes = [j for j in range(len(vocabulario)) if conteos[0, j] > 0]
tabla = pd.DataFrame({
    "palabra": [vocabulario[j] for j in presentes],
    "tf": [conteos[0, j] for j in presentes],
    "df": [df[j] for j in presentes],
    "idf": [round(idf[j], 4) for j in presentes],
    "tf·idf": [round(bruto[0, j], 4) for j in presentes],
    "tf-idf normalizado (L2)": [round(tfidf[0, j], 4) for j in presentes],
})
print(f"Documento 1: {CORPUS['spa'][0]!r} (n = {n} documentos)")
print(tabla.to_string(index=False))
print("norma de la fila sin normalizar:", round(float(np.sqrt((bruto[0] ** 2).sum())), 4))

**¿Qué observas?** `envío` aparece en 2 de 5 documentos: `idf = ln(6/3) + 1 = 1,6931`. `rápido`, que aparece en 1: `ln(6/2) + 1 = 2,0986`. `el`, presente en los cinco: `ln(6/6) + 1 = 1` (el `+ 1` evita que su peso sea cero). Dividiendo cada valor entre la norma de la fila (3,2010) se obtiene la fila normalizada, cuyo cuadrado suma 1. **Ojo:** los valores de una fila TF-IDF **no** son probabilidades ni suman 1.

### La versión del libro no es la de scikit-learn

Manning, Raghavan y Schütze (*Introduction to Information Retrieval*, cap. 6) definen en su versión más simple `idf(t) = log(n / df(t))` y `tf-idf = tf · idf`, **sin** normalizar las filas (el libro presenta además otras variantes de tf y de normalización). Comparamos los dos `idf`. La diferencia de base del logaritmo solo cambia la escala: aquí usamos el logaritmo natural en ambas.

In [ ]:
vocabulario, conteos, df, idf, bruto, tfidf = tfidf_a_mano(CORPUS["spa"])
n = len(CORPUS["spa"])
idf_libro = np.log(n / df)                              # Manning et al.: sin suavizado, sin «+ 1»
comparacion = pd.DataFrame({"df": df, "idf scikit-learn": idf.round(4), "idf libro": idf_libro.round(4)}, index=vocabulario)
print(comparacion.T.to_string())

# Documento 1 con las dos versiones
libro_d1 = conteos[0] * idf_libro
print()
print("Documento 1, versión del libro (tf · log(n/df), sin normalizar):")
print({vocabulario[j]: round(libro_d1[j], 4) for j in range(len(vocabulario)) if conteos[0, j] > 0})
print("Documento 1, versión de scikit-learn (con L2):")
print({vocabulario[j]: round(tfidf[0, j], 4) for j in range(len(vocabulario)) if conteos[0, j] > 0})

# Sin suavizado, scikit-learn usa ln(n/df) + 1; nuestro cálculo también lo reproduce
tv_sin = TfidfVectorizer(token_pattern=r"\w+", smooth_idf=False).fit(CORPUS["spa"])
assert np.allclose(tv_sin.idf_, np.log(n / df) + 1)

**¿Qué observas?** Con la fórmula del libro, `el` vale **0** (aparece en todos los documentos, no distingue nada) y las demás palabras valen entre 0,51 y 1,61. Con la de scikit-learn `el` vale **1** y el resto entre 1,41 y 2,10: el `+ 1` (y el suavizado) desplazan todos los pesos y evitan los ceros. Además scikit-learn normaliza la fila. **Los rankings por palabra son parecidos, los números no**: por eso no conviene comparar valores de TF-IDF entre bibliotecas.

> **Atención:** este «suavizado del IDF» **no** es el suavizado de Laplace de las clases 5 y 8. Aquí solo se evita dividir por cero y anular el peso de palabras presentes en todos los documentos; el suavizado de Laplace suma pseudorrecuentos para que ningún evento no visto tenga probabilidad cero.

---
## 4. Dispersión: `scipy.sparse`

Con un vocabulario de miles de palabras, cada reseña usa unas pocas decenas: **casi todas las celdas son cero**. Guardarlas una a una es un desperdicio. `scipy.sparse` guarda solo los valores no nulos; el formato **CSR** (*compressed sparse row*) usa tres arreglos: `data` (los valores), `indices` (la columna de cada valor) e `indptr` (dónde empieza cada fila). Vamos a medirlo sobre las reseñas brutas.

In [ ]:
def bytes_csr(matriz):
    return matriz.data.nbytes + matriz.indices.nbytes + matriz.indptr.nbytes

filas = []
for idioma, df in resenas.items():
    X_res = CountVectorizer(token_pattern=r"\w+").fit_transform(df["texto"])
    assert sp.issparse(X_res) and X_res.format == "csr"
    denso = X_res.toarray()
    assert denso.nbytes == X_res.shape[0] * X_res.shape[1] * X_res.dtype.itemsize
    filas.append({
        "idioma": idioma, "forma": X_res.shape, "no nulos": X_res.nnz,
        "% no nulos": round(100 * X_res.nnz / (X_res.shape[0] * X_res.shape[1]), 2),
        "CSR (KB)": round(bytes_csr(X_res) / 1024, 1), "denso (KB)": round(denso.nbytes / 1024, 1),
        "veces menos": round(denso.nbytes / bytes_csr(X_res), 1),
    })
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Solo alrededor del 1 % de las celdas es distinto de cero (1,01 % en español, 0,93 % en italiano) y el formato CSR ocupa entre 65 y 71 veces menos memoria que la matriz densa. Con 300 documentos el ahorro es de megabytes; con 100 000 documentos y un vocabulario de 100 000 palabras, una matriz densa de `int64` ocuparía unos 80 GB (100 000 × 100 000 × 8 bytes) y **no cabría** en un portátil: la dispersa sí.

### Efecto de `min_df`, `max_df` y `max_features`

Estos tres parámetros recortan el vocabulario. `min_df=k` descarta las palabras que aparecen en menos de `k` documentos (con un número entero) y `max_df=p` las que aparecen en más de una proporción `p` de documentos. `max_features=m` conserva solo las `m` más frecuentes.

In [ ]:
opciones = {
    "base": {},
    "lowercase=False": {"lowercase": False},
    "min_df=2": {"min_df": 2},
    "min_df=5": {"min_df": 5},
    "max_df=0.5": {"max_df": 0.5},
    "max_df=0.1": {"max_df": 0.1},
    "max_features=500": {"max_features": 500},
    "min_df=2, max_df=0.5": {"min_df": 2, "max_df": 0.5},
}
filas = []
for nombre, parametros in opciones.items():
    fila = {"opción": nombre}
    for idioma, df in resenas.items():
        fila[idioma] = len(CountVectorizer(token_pattern=r"\w+", **parametros).fit(df["texto"]).vocabulary_)
    filas.append(fila)
tabla_vocab = pd.DataFrame(filas)
print(tabla_vocab.to_string(index=False))

base_es, min2_es = tabla_vocab.loc[0, "spa"], tabla_vocab.loc[2, "spa"]
print(f"\nPalabras que aparecen en un solo documento (español): {base_es - min2_es} de {base_es} ({100 * (base_es - min2_es) / base_es:.1f} %)")

ax = tabla_vocab.set_index("opción")[["spa", "ita"]].plot.barh(figsize=(8, 4))
ax.invert_yaxis()
ax.set_xlabel("tamaño del vocabulario")
ax.set_title("Efecto de los parámetros sobre el vocabulario (300 reseñas por idioma)")
plt.tight_layout()
plt.show()

**¿Qué observas?** Dos de cada tres palabras (66,2 % en español) aparecen en un solo documento: `min_df=2` reduce el vocabulario de 1 613 a 545. `max_df=0.5` casi no cambia nada (quita una palabra) porque en un corpus corto y variado casi ninguna palabra llega al 50 % de los documentos; `max_df=0.1` sí quita unas veinte. Las cifras son de este corpus pequeño: en uno grande el efecto de `max_df` es mayor. `lowercase=False` **aumenta** el vocabulario (1 869): `Excelente` y `excelente` pasan a ser dos columnas.

---
## 5. Similitud del coseno

Dos documentos son parecidos si sus vectores **apuntan en la misma dirección**: `cos(u, v) = (u · v) / (‖u‖ ‖v‖)`, con valores entre 0 (ninguna palabra en común, para vectores no negativos) y 1 (misma dirección). Como las filas de `TfidfVectorizer` ya tienen norma 1, el coseno es simplemente el producto escalar. Lo hacemos a mano con tres vectores y lo comprobamos.

In [ ]:
def coseno(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

for idioma in ("spa", "ita"):
    corpus = CORPUS[idioma]
    tfidf_mini = TfidfVectorizer(token_pattern=r"\w+").fit_transform(corpus).toarray()
    conteos_mini = CountVectorizer(token_pattern=r"\w+").fit_transform(corpus).toarray()
    d1, d3, d4 = 0, 2, 3
    for a, b in [(d1, d3), (d3, d4)]:
        print(f"{idioma} d{a+1}-d{b+1}: a mano (TF-IDF) = {coseno(tfidf_mini[a], tfidf_mini[b]):.4f} · con recuentos = {coseno(conteos_mini[a], conteos_mini[b]):.4f}")
    matriz_cos = cosine_similarity(tfidf_mini)
    assert np.isclose(matriz_cos[d3, d4], coseno(tfidf_mini[d3], tfidf_mini[d4]))
    assert np.isclose(matriz_cos[d3, d4], tfidf_mini[d3] @ tfidf_mini[d4])          # filas con norma 1: coseno = producto escalar
    masparecido = np.argsort(-np.where(np.eye(5, dtype=bool), -1, matriz_cos)[d3])[0]
    print(f"   documento más parecido a d3: d{masparecido + 1}")

**¿Qué observas?** `el producto no está bien` y `el producto está muy bien` (d3 y d4) son los más parecidos del corpus (0,707 con TF-IDF, 0,8 con recuentos): comparten `el`, `producto`, `está` y `bien`. La coincidencia de `el` pesa menos con TF-IDF que con recuentos. El coseno **no distingue** `no está bien` de `está bien`: el `no` solo se nota como una palabra más.

### Reseñas parecidas

Sobre las reseñas limpias buscamos, para una de ellas, las tres más parecidas (TF-IDF de palabras, ajustado sobre todo el corpus solo para explorar).

In [ ]:
INDICE_CONSULTA = 5
for idioma, df in resenas.items():
    T = TfidfVectorizer(token_pattern=r"\w+").fit_transform(df["texto_limpio"])
    sim = cosine_similarity(T[INDICE_CONSULTA], T).ravel()
    sim[INDICE_CONSULTA] = -1                                  # excluimos la propia reseña
    print(f"[{idioma}] Consulta ({df.loc[INDICE_CONSULTA, 'etiqueta']}): {df.loc[INDICE_CONSULTA, 'texto_limpio']}")
    for j in np.argsort(-sim)[:3]:
        print(f"   {sim[j]:.3f} ({df.loc[j, 'etiqueta']}): {df.loc[j, 'texto_limpio'][:110]}")
    print()

filas = []
for idioma, df in resenas.items():
    T = TfidfVectorizer(token_pattern=r"\w+").fit_transform(df["texto_limpio"])
    S_todo = cosine_similarity(T)
    np.fill_diagonal(S_todo, -1)
    vecino = S_todo.argmax(axis=1)
    filas.append({"idioma": idioma, "misma etiqueta que su vecino más cercano": round((df["etiqueta"].values[vecino] == df["etiqueta"].values).mean(), 3),
                  "misma etiqueta al azar (3 clases)": round(1 / 3, 3)})
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Los vecinos comparten **palabras**, no necesariamente significado. En español la consulta y sus vecinos comparten la fórmula final `Ahorraos el disgusto` y los tres son negativos; en italiano el vecino más cercano de la consulta positiva (un b&b) es **negativo**, porque comparten `b&b` y `mobili`. Sobre las 300 reseñas, el vecino más cercano tiene la misma etiqueta el 64,7 % de las veces (español) y el 64,3 % (italiano), frente al 33,3 % del azar: la similitud léxica informa, pero no basta.

---
## 6. Experimento guiado: el «gemelo limpio»

**Pregunta.** Cada reseña bruta (`texto`) tiene una versión limpia (`texto_limpio`) que es su «gemelo». Si indexamos las 300 versiones limpias y buscamos, para cada texto bruto, el más parecido por coseno, ¿cuántas veces recuperamos a su gemelo?

**Hipótesis.** Los n-gramas de caracteres son más robustos a las erratas que las palabras completas, así que deberían acertar más.

**Diseño.** Ajustamos el vectorizador sobre las versiones limpias (el «índice»), transformamos con él los textos brutos (consultas) y comparamos: TF-IDF de palabras frente a TF-IDF de n-gramas de caracteres (`char_wb`, 3 a 5). Con dos tipos de consulta: el **texto bruto completo** y una consulta **corta** (solo sus 3 primeras palabras), más difícil.

In [ ]:
def recuperar(df, vectorizador, consulta):
    """Para cada consulta, índice del texto limpio más parecido (el vectorizador se ajusta con los textos limpios)."""
    indice = vectorizador.fit_transform(df["texto_limpio"])
    consultas = vectorizador.transform(consulta)
    similitudes = cosine_similarity(consultas, indice)
    return similitudes.argmax(axis=1), similitudes

def tres_primeras(texto):
    return " ".join(texto.split()[:3])

def crear(tipo):
    if tipo == "palabras":
        return TfidfVectorizer(token_pattern=r"\w+")
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))

resultados, casos = [], {}
for idioma, df in resenas.items():
    gemelo = np.arange(len(df))
    for nombre_consulta, consulta in [("texto bruto completo", df["texto"]), ("3 primeras palabras", df["texto"].map(tres_primeras))]:
        for tipo in ("palabras", "caracteres"):
            elegido, sim = recuperar(df, crear(tipo), consulta)
            aciertos = elegido == gemelo
            resultados.append({"idioma": idioma, "consulta": nombre_consulta, "TF-IDF": "palabras" if tipo == "palabras" else "caracteres (3-5)",
                               "aciertos (de 300)": int(aciertos.sum()),
                               "similitud mínima con el gemelo": round(float(sim[gemelo, gemelo].min()), 3)})
            casos[(idioma, nombre_consulta, tipo)] = (aciertos, elegido)
tabla_gemelo = pd.DataFrame(resultados)
print(tabla_gemelo.to_string(index=False))

**¿Qué observas?** Con el texto bruto completo **los dos aciertan 300 de 300** en ambos idiomas: con el texto entero sobran pistas y no hay diferencia de acierto. Donde los caracteres sí ganan es en el **peor caso**: la similitud mínima con el gemelo sube de 0,568 a 0,670 (español) y de 0,644 a 0,799 (italiano). Con consultas cortas (3 palabras) el resultado se **mezcla**: en español 244 (palabras) frente a 245 (caracteres); en italiano 261 frente a 252, es decir, **ganan las palabras**. La hipótesis se confirma solo a medias: los caracteres dan más margen de seguridad, pero no aciertan más. Un experimento con datos sintéticos y textos cortos no permite generalizar.

Veamos algunos fallos con la consulta corta (español, palabras).

In [ ]:
idioma_ejemplo = "spa"                                   # mostramos el español
nombre_corta = "3 primeras palabras"

filas = []
for idioma in ("spa", "ita"):
    ac_pal, _ = casos[(idioma, nombre_corta, "palabras")]
    ac_car, _ = casos[(idioma, nombre_corta, "caracteres")]
    filas.append({"idioma": idioma, "solo palabras": int((ac_pal & ~ac_car).sum()), "solo caracteres": int((ac_car & ~ac_pal).sum()),
                  "ambos": int((ac_pal & ac_car).sum()), "ninguno": int((~ac_pal & ~ac_car).sum())})
print("Consulta corta: quién acierta")
print(pd.DataFrame(filas).to_string(index=False))

aciertos, elegido = casos[(idioma_ejemplo, nombre_corta, "palabras")]
aciertos_car, _ = casos[(idioma_ejemplo, nombre_corta, "caracteres")]
df = resenas[idioma_ejemplo]
print()
for i in np.where(~aciertos)[0][:4]:
    print(f"Consulta #{i}: {tres_primeras(df.loc[i, 'texto'])!r} · ¿acierta con caracteres? {'sí' if aciertos_car[i] else 'no'}")
    print(f"   gemelo : {df.loc[i, 'texto_limpio'][:90]}")
    print(f"   elegido : {df.loc[elegido[i], 'texto_limpio'][:90]}")

**¿Qué observas?** Con consultas cortas, palabras y caracteres fallan en casos **distintos**: solo aciertan las palabras 14 veces, solo los caracteres 15 (español); 18 y 9 en italiano. Ninguno acierta 41 (español) y 30 (italiano) consultas. Hay dos causas: **arranques genéricos** que no distinguen (`Todo sucio y`, `Muy recomendable para`) y **erratas** dentro de las tres palabras (`Camara de sguridad`), que en este ejemplo tampoco arreglan los caracteres. La representación usa lo que le das; con poca información falla, y los caracteres no son una solución mágica.

---
## 7. Límites y una regla de oro: `fit` solo con el entrenamiento

Las representaciones dispersas tienen límites claros: el **vocabulario crece** con el corpus, el **orden se pierde** (salvo el orden local de los n-gramas), los **sinónimos no se capturan** (`bueno` y `excelente` son columnas independientes; los *embeddings* de la clase 10 atacan esto) y todo depende del **preprocesamiento** (clase 3): un acento perdido crea una palabra nueva.

Una consecuencia práctica: el vectorizador **aprende** su vocabulario y sus `idf` con `fit`; con `transform` solo los aplica. Si lo ajustas con **todos** los datos antes de dividir en entrenamiento y prueba, la prueba «se cuela» en el modelo (fuga de información, en detalle en la clase 5). Miramos qué vocabulario tiene un `fit` solo con el 80 % de las reseñas y qué palabras de las reseñas de prueba le son desconocidas.

In [ ]:
filas = []
for idioma, df in resenas.items():
    entrenamiento, prueba = train_test_split(df["texto"], test_size=0.2, random_state=42)
    vec = CountVectorizer(token_pattern=r"\w+")
    vec.fit(entrenamiento)                                   # fit: solo con el entrenamiento
    X_prueba = vec.transform(prueba)                         # transform: aplica el vocabulario ya aprendido
    assert X_prueba.shape[1] == len(vec.vocabulary_)         # mismas columnas que el entrenamiento

    tokens_prueba = [t for texto in prueba for t in tokenizar(texto)]
    desconocidos = [t for t in tokens_prueba if t not in vec.vocabulary_]
    filas.append({
        "idioma": idioma, "vocabulario de entrenamiento": len(vec.vocabulary_),
        "tokens de prueba": len(tokens_prueba), "tokens desconocidos": len(desconocidos),
        "% desconocidos": round(100 * len(desconocidos) / len(tokens_prueba), 1),
    })
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Alrededor de una cuarta parte de los tokens de la prueba (24,2 % en español y 24,0 % en italiano) **no existen** en el vocabulario de entrenamiento y `transform` los ignora: no generan ninguna columna. Es lo que ocurrirá con textos nuevos en producción, y por eso la evaluación debe hacerse así. Si ajustaras el vectorizador con todo el corpus, esas palabras sí tendrían columna y `idf`, y el resultado sería optimista.

## 8. Resumen y siguiente paso

- La **bolsa de palabras** convierte cada documento en un vector de recuentos; `CountVectorizer` lo hace con parámetros (`lowercase`, `token_pattern`, `stop_words`, `ngram_range`, `min_df`, `max_df`, `max_features`). El patrón por defecto descarta los tokens de un carácter.
- Los **n-gramas** recuperan orden local (`no es bueno`) y robustez ante erratas (caracteres), al precio de un vocabulario mucho mayor.
- **TF-IDF de scikit-learn:** `idf = ln((1 + n) / (1 + df)) + 1` y filas con norma L2; no coincide con `log(n/df)` del libro y **no es una probabilidad**. Ese suavizado no es el de Laplace.
- Las matrices son **dispersas** (≈ 1 % de celdas no nulas en las reseñas) y `scipy.sparse` (CSR) las guarda con una fracción de la memoria.
- El **coseno** compara documentos; con TF-IDF de filas normalizadas es un producto escalar. Mide palabras compartidas, no significado.
- El vectorizador se ajusta (`fit`) **solo con el entrenamiento**.

**Siguiente:** en la clase 5, *Clasificación de textos*, usarás estas representaciones dentro de una `Pipeline` con Naive Bayes, regresión logística y SVM. Antes, haz el notebook de práctica de esta clase.